### Let's see the optimal approach to create the best RAG pipeline 

In [ ]:
# 1. Let's Read The PDF 
from pathlib import Path 
from langchain_community.document_loaders import PyMuPDFLoader

BASE_DIR = Path.cwd().resolve().parent
file_path = BASE_DIR / "config" / "data" / "processed" / "AttentionYou.pdf"

loader = PyMuPDFLoader(str(file_path))
text_document = loader.load()

# 5. Let's perform the embeddings 
from langchain_huggingface import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(
    model_name = "BAAI/bge-large-en-v1.5"
)

# 2. Let's clean the text content from the document 
import re 
def clean_document(doc) :
    text = doc.page_content
    # Remove excessive spaces but preserve newlines
    text = re.sub(r'[ \t]+', ' ', text)

    # Collapse 3+ newlines into 2
    text = re.sub(r'\n{3,}', '\n\n', text)

    # Remove standalone page numbers
    text = re.sub(r'\n\d+\n', '\n', text)

    doc.page_content = text 
    
    return doc 

cleaned_text = [clean_document(doc) for doc in text_document]


# 4. Let's perform chunking 
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = (RecursiveCharacterTextSplitter.from_tiktoken_encoder(
    chunk_size=800,
    chunk_overlap=150,
    separators=[
        "\n\n",
        "\n",
        ". ",
        "? "
        "! "
        " "
    ]
))

chunks = splitter.split_documents(cleaned_text)


# Let's create our Vector database 
from langchain_community.vectorstores import Chroma

db = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    persist_directory="./chroma_db",
    collection_name="attention"
)


# 6. Let's perform our retrieval mechanism 
retriever = db.as_retriever(
    search_type = "similarity",
    search_kwargs={
        "k":10
    }
)

# 7. Let's test our retrievel mechanism
query = "what is the scaled dot product ?"
top_k_chunks = retriever.invoke(query)

# Let's see the results 
for clear_chunk in top_k_chunks :
    text = clear_chunk.page_content
    text = re.sub(r'\s+', ' ', text)
    text = re.sub(r'\n\d+\n', '\n', text)
    print(text)


C:\Users\mahen\AppData\Local\Temp\ipykernel_13292\1579550199.py:3: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyMuPDFLoader
d:\AI-Restart-2026\Complete GEN AI Journey - Notes\DataIngestion_Pipeline\DataIngestionDemo\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


[Document(metadata={'producer': 'PyPDF2', 'creator': '', 'creationdate': '', 'source': 'D:\\AI-Restart-2026\\Complete GEN AI Journey - Notes\\DataIngestion_Pipeline\\config\\data\\processed\\AttentionYou.pdf', 'file_path': 'D:\\AI-Restart-2026\\Complete GEN AI Journey - Notes\\DataIngestion_Pipeline\\config\\data\\processed\\AttentionYou.pdf', 'total_pages': 11, 'format': 'PDF 1.3', 'title': 'Attention is All you Need', 'author': 'Ashish Vaswani, Noam Shazeer, Niki Parmar, Jakob Uszkoreit, Llion Jones, Aidan N. Gomez, Łukasz Kaiser, Illia Polosukhin', 'subject': 'Neural Information Processing Systems http://nips.cc/', 'keywords': '', 'moddate': '2018-02-12T21:22:10-08:00', 'trapped': '', 'modDate': "D:20180212212210-08'00'", 'creationDate': '', 'page': 0}, page_content='Attention Is All You Need\nAshish Vaswani∗\nGoogle Brain\navaswani@google.com\nNoam Shazeer∗\nGoogle Brain\nnoam@google.com\nNiki Parmar∗\nGoogle Research\nnikip@google.com\nJakob Uszkoreit∗\nGoogle Research\nusz@google

In [25]:
# 2. Let's clean the text content from the document 
import re 
def clean_document(doc) :
    text = doc.page_content
    # Remove excessive spaces but preserve newlines
    text = re.sub(r'[ \t]+', ' ', text)

    # Collapse 3+ newlines into 2
    text = re.sub(r'\n{3,}', '\n\n', text)

    # Remove standalone page numbers
    text = re.sub(r'\n\d+\n', '\n', text)

    doc.page_content = text 
    
    return doc 

cleaned_text = [clean_document(doc) for doc in text_document]


In [26]:
# 3. Here is our cleaned text 
cleaned_text


[Document(metadata={'producer': 'PyPDF2', 'creator': '', 'creationdate': '', 'source': 'D:\\AI-Restart-2026\\Complete GEN AI Journey - Notes\\DataIngestion_Pipeline\\config\\data\\processed\\AttentionYou.pdf', 'file_path': 'D:\\AI-Restart-2026\\Complete GEN AI Journey - Notes\\DataIngestion_Pipeline\\config\\data\\processed\\AttentionYou.pdf', 'total_pages': 11, 'format': 'PDF 1.3', 'title': 'Attention is All you Need', 'author': 'Ashish Vaswani, Noam Shazeer, Niki Parmar, Jakob Uszkoreit, Llion Jones, Aidan N. Gomez, Łukasz Kaiser, Illia Polosukhin', 'subject': 'Neural Information Processing Systems http://nips.cc/', 'keywords': '', 'moddate': '2018-02-12T21:22:10-08:00', 'trapped': '', 'modDate': "D:20180212212210-08'00'", 'creationDate': '', 'page': 0}, page_content='Attention Is All You Need Ashish Vaswani∗ Google Brain avaswani@google.com Noam Shazeer∗ Google Brain noam@google.com Niki Parmar∗ Google Research nikip@google.com Jakob Uszkoreit∗ Google Research usz@google.com Llion J

In [27]:
# 4. Let's perform chunking 
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = (RecursiveCharacterTextSplitter.from_tiktoken_encoder(
    chunk_size=800,
    chunk_overlap=150,
    separators=[
        "\n\n",
        "\n",
        ". ",
        "? "
        "! "
        " "
    ]
))

chunks = splitter.split_documents(cleaned_text)

In [28]:
# Let's see the chunks 
chunks

[Document(metadata={'producer': 'PyPDF2', 'creator': '', 'creationdate': '', 'source': 'D:\\AI-Restart-2026\\Complete GEN AI Journey - Notes\\DataIngestion_Pipeline\\config\\data\\processed\\AttentionYou.pdf', 'file_path': 'D:\\AI-Restart-2026\\Complete GEN AI Journey - Notes\\DataIngestion_Pipeline\\config\\data\\processed\\AttentionYou.pdf', 'total_pages': 11, 'format': 'PDF 1.3', 'title': 'Attention is All you Need', 'author': 'Ashish Vaswani, Noam Shazeer, Niki Parmar, Jakob Uszkoreit, Llion Jones, Aidan N. Gomez, Łukasz Kaiser, Illia Polosukhin', 'subject': 'Neural Information Processing Systems http://nips.cc/', 'keywords': '', 'moddate': '2018-02-12T21:22:10-08:00', 'trapped': '', 'modDate': "D:20180212212210-08'00'", 'creationDate': '', 'page': 0}, page_content='Attention Is All You Need Ashish Vaswani∗ Google Brain avaswani@google.com Noam Shazeer∗ Google Brain noam@google.com Niki Parmar∗ Google Research nikip@google.com Jakob Uszkoreit∗ Google Research usz@google.com Llion J

In [ ]:
# 5. Let's perform the embeddings 
from langchain_huggingface import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(
    model_name = "BAAI/bge-large-en-v1.5"
)

d:\AI-Restart-2026\Complete GEN AI Journey - Notes\DataIngestion_Pipeline\DataIngestionDemo\Lib\site-packages\huggingface_hub\file_download.py:138: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\mahen\.cache\huggingface\hub\models--BAAI--bge-large-en-v1.5. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Loading weights: 100%|██████████| 391/391 [00:00<00:00

In [29]:
# Let's create our Vector database 
from langchain_community.vectorstores import Chroma

db = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    persist_directory="./chroma_db",
    collection_name="attention"
)

In [ ]:
# 6. Let's perform our retrieval mechanism 
retriever = db.as_retriever(
    search_type = "similarity",
    search_kwargs={
        "k":10
    }
)

# 7. Let's test our retrievel mechanism
query = "what is the scaled dot product ?"
top_k_chunks = retriever.invoke(query)

# Let's see the results 
for clear_chunk in top_k_chunks :
    text = clear_chunk.page_content
    text = re.sub(r'\s+', ' ', text)
    text = re.sub(r'\n\d+\n', '\n', text)
    print(text)

In [ ]:
# 7. Let's test our retrievel mechanism
query = "what is the scaled dot product ?"
top_k_chunks = retriever.invoke(query)

In [42]:
# Let's see the results 
for clear_chunk in top_k_chunks :
    text = clear_chunk.page_content
    text = re.sub(r'\s+', ' ', text)
    text = re.sub(r'\n\d+\n', '\n', text)
    print(text)
    

Scaled Dot-Product Attention Multi-Head Attention Figure 2: (left) Scaled Dot-Product Attention. (right) Multi-Head Attention consists of several attention layers running in parallel. query with all keys, divide each by √dk, and apply a softmax function to obtain the weights on the values. In practice, we compute the attention function on a set of queries simultaneously, packed together into a matrix Q. The keys and values are also packed together into matrices K and V . We compute the matrix of outputs as: Attention(Q, K, V ) = softmax(QKT √dk )V (1) The two most commonly used attention functions are additive attention [2], and dot-product (multi- plicative) attention. Dot-product attention is identical to our algorithm, except for the scaling factor of 1 √dk . Additive attention computes the compatibility function using a feed-forward network with a single hidden layer. While the two are similar in theoretical complexity, dot-product attention is much faster and more space-efﬁcient i

In [43]:
# 8. Let's do the reranking 
from sentence_transformers import CrossEncoder

reranker = CrossEncoder(
    "BAAI/bge-reranker-base"
)

d:\AI-Restart-2026\Complete GEN AI Journey - Notes\DataIngestion_Pipeline\DataIngestionDemo\Lib\site-packages\huggingface_hub\file_download.py:138: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\mahen\.cache\huggingface\hub\models--BAAI--bge-reranker-base. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Loading weights: 100%|██████████| 201/201 [00:00<00:00

In [44]:
# Let's invoke the query 
docs = retriever.invoke(query)

In [48]:
# calculating the scores for each chunk againts the query 
pairs = [(query,doc.page_content) for doc in docs]
scores = reranker.predict(pairs)

In [49]:
# Let's see the scores 
scores

array([9.6663409e-01, 9.5301783e-01, 4.0092586e-05, 3.6995749e-03,
       6.7988716e-05, 8.2539308e-01, 2.3087757e-04, 3.7323382e-05,
       1.5422980e-04, 3.7302350e-05], dtype=float32)

### Clear And Most Accurate Context Generation 

In [58]:
# sorting the chunks based on the scores 
reranked = sorted(zip(docs, scores),key=lambda x: x[1],reverse=True)
# let's extract top documents
top_doc = [doc for doc,score in reranked[:5]]
# Let's create the abosolute context 
context = "\n\n".join(
    doc.page_content
    for doc in top_doc
)

In [ ]:
# Here is our final context 
context

'Scaled Dot-Product Attention Multi-Head Attention Figure 2: (left) Scaled Dot-Product Attention. (right) Multi-Head Attention consists of several attention layers running in parallel. query with all keys, divide each by √dk, and apply a softmax function to obtain the weights on the values. In practice, we compute the attention function on a set of queries simultaneously, packed together into a matrix Q. The keys and values are also packed together into matrices K and V . We compute the matrix of outputs as: Attention(Q, K, V ) = softmax(QKT √dk )V (1) The two most commonly used attention functions are additive attention [2], and dot-product (multi- plicative) attention. Dot-product attention is identical to our algorithm, except for the scaling factor of 1 √dk . Additive attention computes the compatibility function using a feed-forward network with a single hidden layer. While the two are similar in theoretical complexity, dot-product attention is much faster and more space-efﬁcient 

In [70]:
# 9. Let's see the prompt designing 
prompt = f"""
You are an expert researcher.

Your task is to answer ONLY using the supplied context.

Instructions:
- Think carefully before answering.
- Extract key information from the context.
- Explain technical concepts in detail.
- Quote important phrases when necessary.
- If the answer is not present in the context, say so.
- Do not use outside knowledge.

CONTEXT:
{context}

QUESTION:
{query}

Provide:
1. Direct Answer
2. Technical Explanation
3. Key Takeaways
"""

In [66]:
from openai import OpenAI
import os

client = OpenAI(
    base_url="https://integrate.api.nvidia.com/v1",
    api_key="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"
)

completion = client.chat.completions.create(
    model="nvidia/nemotron-3-ultra-550b-a55b",
    messages=[
        {
            "role": "system",
            "content": """
            You are a retrieval-augmented AI assistant.

            Answer ONLY from the provided context.
            Do not hallucinate.
            If information is missing, explicitly state that.
            """
        },
        {
            "role": "user",
            "content": prompt
        }
    ],
    temperature=0.1,
    top_p=0.9,
    max_tokens=4096,
    stream=True
)

answer = ""

for chunk in completion:

    if not chunk.choices:
        continue

    if chunk.choices[0].delta.content:
        token = chunk.choices[0].delta.content
        answer += token
        print(token, end="", flush=True)

print("\n\nFinal Answer:")
print(answer)

Based on the provided context, **Scaled Dot-Product Attention** is an attention mechanism where:

1. **Inputs**: Queries and keys of dimension *dₖ*, and values of dimension *dᵥ*.
2. **Computation**:
   - Compute dot products of the query with all keys.
   - Divide each dot product by √*dₖ* (the scaling factor).
   - Apply a softmax function to obtain weights on the values.
3. **Formula**:
   \[
   \text{Attention}(Q, K, V) = \text{softmax}\left(\frac{QK^T}{\sqrt{d_k}}\right)V
   \]
4. **Purpose of scaling**: For large *dₖ*, dot products grow large in magnitude (variance *dₖ*), pushing the softmax into regions with extremely small gradients. Scaling by 1/√*dₖ* counteracts this effect.

The context also notes that dot-product attention is faster and more space-efficient than additive attention because it can use highly optimized matrix multiplication.

Final Answer:
Based on the provided context, **Scaled Dot-Product Attention** is an attention mechanism where:

1. **Inputs**: Queries an

In [71]:
# Let's test with our local qwin model 
from langchain_ollama import ChatOllama

llm = ChatOllama(
    model="qwen3:8b",
    temperature=0.3
)

response = llm.invoke(prompt)

In [ ]:
# cleaning 
text = response.text
text = re.sub(r'\s+', ' ', text)
text = re.sub(r'\n\d+\n', '\n', text)
final_answer = text

In [75]:
# final answer 
final_answer

'1. **Direct Answer** Scaled Dot-Product Attention is a mechanism in the Transformer model where the dot product of queries (Q) and keys (K) is scaled by dividing by the square root of the dimension of keys ($ \\sqrt{d_k} $) before applying the softmax function. This scaling ensures numerical stability and prevents gradients from vanishing during training. --- 2. **Technical Explanation** - **Computation**: The attention mechanism calculates the compatibility between queries and keys using their dot product. For example, given matrices $ Q \\in \\mathbb{R}^{n \\times d_k} $, $ K \\in \\mathbb{R}^{m \\times d_k} $, and $ V \\in \\mathbb{R}^{m \\times d_v} $, the scaled dot product is computed as: $$ \\text{Attention}(Q, K, V) = \\text{softmax}\\left(\\frac{QK^T}{\\sqrt{d_k}}\\right)V $$ Here, $ \\frac{QK^T}{\\sqrt{d_k}} $ normalizes the dot products to prevent large values. - **Scaling Factor**: Without scaling, the dot products grow linearly with $ d_k $, causing the softmax function t

In [1]:
# Let's test with our local qwin model 
from langchain_ollama import ChatOllama

llm = ChatOllama(
    model="qwen3:8b",
    temperature=0.3
)

query = "Who is mahendra singh dhoni ?"
response = llm.invoke(query)

d:\AI-Restart-2026\Complete GEN AI Journey - Notes\DataIngestion_Pipeline\DataIngestionDemo\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
response

AIMessage(content='**Mahendra Singh Dhoni** (born 7 July 1981) is a former Indian cricketer, widely regarded as one of the greatest captains in the history of cricket. Here\'s an overview of his career and legacy:\n\n### **Key Highlights:**\n1. **Early Career:**\n   - Born in Ranchi, Jharkhand, Dhoni began his cricketing journey with the Indian Railways team and later joined the Delhi Daredevils in the Indian Premier League (IPL) in 2008.\n   - His calm demeanor and exceptional wicketkeeping skills earned him a place in the Indian national team.\n\n2. **International Career:**\n   - **Captaincy:** Dhoni captained the Indian team from 2007 to 2020, leading the team to numerous successes. He is the **first Indian captain** to win all three major ICC trophies:\n     - **T20 World Cup (2007, 2021)**\n     - **ODI World Cup (2011)**\n     - **ICC Champions Trophy (2013)**\n   - **Batting Style:** Known as the "Captain Cool," he was a reliable lower-order batter, often delivering crucial run

In [3]:
query = "Who is Demon king?"
response = llm.invoke(query)

In [10]:
import time
from langchain_ollama import ChatOllama

from langchain_ollama import ChatOllama

llm = ChatOllama(
    model="qwen3:8b",
    base_url="http://localhost:11434",
    request_timeout=300,
)

question = "Explain Retrieval Augmented Generation in simple words."

start = time.perf_counter()

response = llm.invoke(question)

end = time.perf_counter()

print(response.content)
print("\nTime taken:", round(end - start, 2), "seconds")

Retrieval Augmented Generation (RAG) is like a two-step process that helps a computer answer questions more accurately. Here's how it works in simple terms:

1. **Retrieval**: The computer first searches for relevant information in a database or collection of documents (like a library or encyclopedia). This step is like looking up facts or answers in a book.

2. **Generation**: Once it finds the right information, the computer uses that data to create a clear, human-like answer. This is like a teacher using a textbook to explain a topic in a way that’s easy to understand.

**Why it’s helpful**:  
RAG ensures the answer is based on real facts (from the database) instead of made-up information. It’s especially useful when accuracy matters, like answering questions about science, history, or customer service. Think of it as combining a "lookup tool" with a "smart explainer" to give better, more trustworthy responses.

Time taken: 54.55 seconds


In [5]:
response

AIMessage(content='', additional_kwargs={}, response_metadata={'model': 'qwen3:8b', 'created_at': '2026-06-14T09:26:48.6318953Z', 'done': True, 'done_reason': 'length', 'total_duration': 40737057600, 'load_duration': 533426600, 'prompt_eval_count': 21, 'prompt_eval_duration': 623152000, 'eval_count': 256, 'eval_duration': 22865401000, 'logprobs': None, 'model_name': 'qwen3:8b', 'model_provider': 'ollama'}, id='lc_run--019ec573-df2e-7f81-b948-77ad2d5879d5-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 21, 'output_tokens': 256, 'total_tokens': 277})

In [6]:
llm.invoke("who is maheendra singh dhoni")

AIMessage(content='', additional_kwargs={}, response_metadata={'model': 'qwen3:8b', 'created_at': '2026-06-14T09:28:56.8025042Z', 'done': True, 'done_reason': 'length', 'total_duration': 25288694300, 'load_duration': 552832900, 'prompt_eval_count': 19, 'prompt_eval_duration': 831629000, 'eval_count': 256, 'eval_duration': 23848569000, 'logprobs': None, 'model_name': 'qwen3:8b', 'model_provider': 'ollama'}, id='lc_run--019ec576-1026-76d2-b567-e481dcae11d4-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 19, 'output_tokens': 256, 'total_tokens': 275})

In [11]:
import time
from langchain_ollama import ChatOllama

from langchain_ollama import ChatOllama

llm = ChatOllama(
    model="gemma3:4b",
    base_url="http://localhost:11434",
    num_predict=256,
    keep_alive="30m",
    request_timeout=300,
)

question = "Explain Retrieval Augmented Generation in simple words."

answer = llm.invoke(question)

In [12]:
answer

AIMessage(content="Okay, let's break down Retrieval Augmented Generation (RAG) simply:\n\n**Imagine you’re asking a really smart friend for help with a complicated question.**\n\n* **Traditional Language Models (like ChatGPT) are like your friend relying *only* on what they've already learned during their training.** They can give an answer, but sometimes that answer is just a guess based on patterns in the massive amount of text they’ve read.  It might be confident but wrong because it hasn't directly been told about the specific information you need.\n\n* **RAG is like your friend doing this:**\n    1. **Retrieval (Finding Relevant Info):** First, your friend quickly searches their notes, books, or a relevant database to find *specific* documents or snippets that relate to your question.  Think of it as searching for the most helpful pieces of information.\n\n    2. **Generation (Answering Based on Found Info):** Then, they use those specific pieces of information *along with* their 